# 1.4 数据标注 (Data Annotation)

数据标注为模型提供监督信号，标注质量直接影响模型性能，尤其在对齐训练中至关重要。

本节涵盖：
- 人工标注
- LLM辅助标注
- 主动学习
- 标注质量控制

## 1. 人工标注

**目的**：获取高质量、可靠的标注数据

**基本原理**：雇佣专业标注人员对数据进行标注，包括偏好排序、质量评分、安全性判断等，是RLHF奖励模型训练数据的主要来源。

**标注类型**：
- **偏好排序**：对同一提示的多个回复进行质量排序（用于RLHF/DPO）
- **质量评分**：对回复质量进行1-5分评分
- **安全性判断**：判断回复是否安全/有害
- **事实性判断**：判断回复是否包含事实错误

**产业实践**：InstructGPT使用了约40名标注员进行偏好数据收集，每条数据由多个标注员独立标注后取多数意见。

In [ ]:
import random
from collections import Counter

random.seed(42)

class HumanAnnotationSimulator:
    def __init__(self, n_annotators=5, agreement_threshold=0.6):
        self.n_annotators = n_annotators
        self.agreement_threshold = agreement_threshold

    def simulate_preference_annotation(self, prompt, responses, true_quality):
        annotations = []
        for ann_id in range(self.n_annotators):
            noisy_quality = true_quality + [random.gauss(0, 0.15) for _ in responses]
            ranking = sorted(range(len(responses)), key=lambda i: -noisy_quality[i])
            annotations.append(ranking)
        return annotations

    def compute_inter_annotator_agreement(self, annotations):
        n = len(annotations)
        if n < 2:
            return 1.0
        agreements = 0
        total = 0
        for i in range(n):
            for j in range(i + 1, n):
                rank_i = annotations[i]
                rank_j = annotations[j]
                pairs = 0
                concordant = 0
                for a in range(len(rank_i)):
                    for b in range(a + 1, len(rank_i)):
                        pairs += 1
                        pos_a_i = rank_i.index(a)
                        pos_b_i = rank_i.index(b)
                        pos_a_j = rank_j.index(a)
                        pos_b_j = rank_j.index(b)
                        if (pos_a_i - pos_b_i) * (pos_a_j - pos_b_j) > 0:
                            concordant += 1
                agreements += concordant / pairs if pairs > 0 else 0
                total += 1
        return agreements / total if total > 0 else 0.0

    def aggregate_annotations(self, annotations):
        n_responses = len(annotations[0])
        borda_scores = [0] * n_responses
        for ranking in annotations:
            for rank, resp_idx in enumerate(ranking):
                borda_scores[resp_idx] += (n_responses - rank)
        final_ranking = sorted(range(n_responses), key=lambda i: -borda_scores[i])
        return final_ranking, borda_scores

simulator = HumanAnnotationSimulator(n_annotators=5)

prompt = "Explain quantum computing in simple terms"
responses = [
    "Quantum computers use qubits that can be 0 and 1 at the same time.",
    "Quantum computing is a type of computing that uses quantum mechanics.",
    "QCs leverage superposition and entanglement for parallel computation.",
]
true_quality = [0.9, 0.4, 0.7]

annotations = simulator.simulate_preference_annotation(prompt, responses, true_quality)
agreement = simulator.compute_inter_annotator_agreement(annotations)
final_ranking, borda_scores = simulator.aggregate_annotations(annotations)

print('=== Human Annotation Simulation ===')
print(f'Prompt: {prompt}')
print(f'\nResponses and true quality:')
for i, (resp, qual) in enumerate(zip(responses, true_quality)):
    print(f'  [{i}] (quality={qual:.1f}): {resp}')

print(f'\nAnnotator rankings (best -> worst):')
for ann_id, ranking in enumerate(annotations):
    print(f'  Annotator {ann_id}: {ranking}')

print(f'\nInter-annotator agreement (Kendall W): {agreement:.3f}')
print(f'Borda count scores: {borda_scores}')
print(f'Final ranking: {final_ranking} (best -> worst)')
print(f'\nBest response: [{final_ranking[0]}] {responses[final_ranking[0]]}')

## 2. LLM辅助标注

**目的**：降低标注成本，提高标注效率

**基本原理**：使用强模型（如GPT-4）对数据进行初步标注或提供标注参考，人工仅需审核和修正，大幅提升标注效率。这种方法也称为LLM-as-a-Judge。

**应用场景**：
- 生成SFT训练数据的初始回复
- 对多个回复进行质量排序
- 检测回复中的安全问题
- 评估回复的事实准确性

In [ ]:
import random

random.seed(42)

class LLMAssistedAnnotator:
    def __init__(self):
        self.criteria = {
            'helpfulness': {
                'prompt': 'Rate how helpful this response is (1-5)',
                'weight': 0.4,
            },
            'accuracy': {
                'prompt': 'Rate how factually accurate this response is (1-5)',
                'weight': 0.3,
            },
            'safety': {
                'prompt': 'Rate how safe this response is (1-5)',
                'weight': 0.3,
            },
        }

    def llm_judge(self, prompt, response, criterion):
        base_scores = {
            'helpfulness': len(response) / 100,
            'accuracy': 3.0 + random.gauss(0, 0.5),
            'safety': 4.0 + random.gauss(0, 0.3),
        }
        score = max(1, min(5, base_scores.get(criterion, 3.0)))
        return round(score, 1)

    def annotate(self, prompt, response):
        scores = {}
        for criterion, config in self.criteria.items():
            scores[criterion] = self.llm_judge(prompt, response, criterion)
        weighted_score = sum(
            scores[c] * self.criteria[c]['weight'] for c in self.criteria
        )
        return {'scores': scores, 'weighted_score': round(weighted_score, 2)}

    def compare_responses(self, prompt, response_a, response_b):
        ann_a = self.annotate(prompt, response_a)
        ann_b = self.annotate(prompt, response_b)
        if ann_a['weighted_score'] > ann_b['weighted_score']:
            winner = 'A'
        elif ann_b['weighted_score'] > ann_a['weighted_score']:
            winner = 'B'
        else:
            winner = 'Tie'
        return winner, ann_a, ann_b

annotator = LLMAssistedAnnotator()

prompt = "What is machine learning?"
responses = [
    "Machine learning is a branch of AI that enables systems to learn from data and improve over time without explicit programming.",
    "ML is AI stuff.",
    "Machine learning algorithms build models based on sample data to make predictions or decisions without being explicitly programmed to do so.",
]

print('=== LLM-Assisted Annotation Demo ===')
print(f'Prompt: {prompt}\n')

for i, resp in enumerate(responses):
    result = annotator.annotate(prompt, resp)
    print(f'Response {i}: "{resp[:60]}..."')
    print(f'  Scores: {result["scores"]}')
    print(f'  Weighted: {result["weighted_score"]}\n')

winner, ann_a, ann_b = annotator.compare_responses(prompt, responses[0], responses[1])
print(f'Comparison: Response 0 vs Response 1')
print(f'  Response 0 score: {ann_a["weighted_score"]}')
print(f'  Response 1 score: {ann_b["weighted_score"]}')
print(f'  Winner: {winner}')

## 3. 主动学习

**目的**：选择最有信息量的数据进行标注

**基本原理**：训练一个初始模型，让模型挑选其最不确定的样本交由人工标注，以最少标注成本获得最大模型提升。

**不确定性度量**：
- **熵**：H(p) = -Σ p_i log(p_i)，熵越高越不确定
- **边缘采样**：选择正负类预测概率最接近0.5的样本
- **方差**：多次dropout前向传播的预测方差

In [ ]:
import numpy as np

np.random.seed(42)

class ActiveLearningSelector:
    def __init__(self, n_classes=3):
        self.n_classes = n_classes

    def entropy(self, probs):
        probs = np.clip(probs, 1e-10, 1.0)
        return -np.sum(probs * np.log(probs), axis=-1)

    def margin(self, probs):
        sorted_probs = np.sort(probs, axis=-1)
        return sorted_probs[:, -1] - sorted_probs[:, -2]

    def select_uncertain(self, probs, n_select=5, strategy='entropy'):
        if strategy == 'entropy':
            scores = self.entropy(probs)
            return np.argsort(-scores)[:n_select]
        elif strategy == 'margin':
            margins = self.margin(probs)
            return np.argsort(margins)[:n_select]
        elif strategy == 'random':
            return np.random.choice(len(probs), n_select, replace=False)

n_samples = 50
n_classes = 3

probs = np.zeros((n_samples, n_classes))
for i in range(n_samples):
    if i < 15:
        p = np.random.dirichlet([10, 1, 1])
    elif i < 30:
        p = np.random.dirichlet([1, 10, 1])
    else:
        p = np.random.dirichlet([1, 1, 1])
    probs[i] = p

selector = ActiveLearningSelector(n_classes)

print('=== Active Learning Sample Selection ===')
print(f'Pool: {n_samples} unlabeled samples, {n_classes} classes')

for strategy in ['entropy', 'margin', 'random']:
    selected = selector.select_uncertain(probs, n_select=5, strategy=strategy)
    selected_probs = probs[selected]
    avg_entropy = selector.entropy(selected_probs).mean()
    avg_margin = selector.margin(selected_probs).mean()
    print(f'\n{strategy.upper()} strategy:')
    print(f'  Selected indices: {selected}')
    print(f'  Avg entropy: {avg_entropy:.3f}, Avg margin: {avg_margin:.3f}')
    for idx in selected[:3]:
        print(f'    Sample {idx}: probs={probs[idx].round(3)}, entropy={selector.entropy(probs[idx:idx+1])[0]:.3f}')

print(f'\nInsight: Entropy/margin strategies select uncertain samples (dirichlet[1,1,1]),')
print(f'which provide the most information for improving the model.')

## 4. 标注质量控制

**目的**：确保标注数据的一致性和准确性

**基本原理**：通过多人标注一致性检验（Cohen's Kappa）、标注规范迭代、金标集抽检等机制保证标注质量。

**关键指标**：
- **Cohen's Kappa**：衡量两个标注员之间的一致性，校正随机一致性
- **Fleiss' Kappa**：衡量多个标注员之间的一致性
- **标注-金标一致率**：标注员与金标答案的一致率

In [ ]:
import numpy as np

np.random.seed(42)

def cohen_kappa(labels1, labels2, n_classes=3):
    n = len(labels1)
    confusion = np.zeros((n_classes, n_classes))
    for a, b in zip(labels1, labels2):
        confusion[a][b] += 1
    p_observed = np.trace(confusion) / n
    p_expected = sum(
        (confusion[i, :].sum() / n) * (confusion[:, i].sum() / n)
        for i in range(n_classes)
    )
    if p_expected == 1.0:
        return 1.0
    return (p_observed - p_expected) / (1 - p_expected)

def fleiss_kappa(annotations, n_classes=3):
    n_items = len(annotations)
    n_annotators = len(annotations[0])
    agreement_per_item = []
    for item_anns in annotations:
        counts = np.bincount(item_anns, minlength=n_classes)
        agree = (counts * (counts - 1)).sum() / (n_annotators * (n_annotators - 1))
        agreement_per_item.append(agree)
    p_mean = np.mean(agreement_per_item)
    class_probs = np.zeros(n_classes)
    for item_anns in annotations:
        counts = np.bincount(item_anns, minlength=n_classes)
        class_probs += counts / (n_items * n_annotators)
    p_expected = (class_probs ** 2).sum()
    if p_expected == 1.0:
        return 1.0
    return (p_mean - p_expected) / (1 - p_expected)

n_items = 100
n_classes = 3
true_labels = np.random.randint(0, n_classes, n_items)

ann1 = np.clip(true_labels + np.random.choice([-1, 0, 0, 0, 1], n_items), 0, n_classes - 1)
ann2 = np.clip(true_labels + np.random.choice([-1, 0, 0, 0, 1], n_items), 0, n_classes - 1)
ann3 = np.clip(true_labels + np.random.choice([-1, 0, 0, 1], n_items), 0, n_classes - 1)

kappa_12 = cohen_kappa(ann1, ann2, n_classes)
kappa_13 = cohen_kappa(ann1, ann3, n_classes)
kappa_23 = cohen_kappa(ann2, ann3, n_classes)

all_annotations = list(zip(ann1, ann2, ann3))
fleiss = fleiss_kappa(all_annotations, n_classes)

print('=== Annotation Quality Control ===')
print(f'Items: {n_items}, Classes: {n_classes}, Annotators: 3')
print(f'\nPairwise Cohen\'s Kappa:')
print(f'  Annotator 1 vs 2: {kappa_12:.3f}')
print(f'  Annotator 1 vs 3: {kappa_13:.3f}')
print(f'  Annotator 2 vs 3: {kappa_23:.3f}')
print(f'\nFleiss\' Kappa (all annotators): {fleiss:.3f}')

print(f'\n--- Interpretation ---')
print(f'  < 0.20: Poor agreement')
print(f'  0.21-0.40: Fair agreement')
print(f'  0.41-0.60: Moderate agreement')
print(f'  0.61-0.80: Substantial agreement')
print(f'  0.81-1.00: Almost perfect agreement')

accuracy_1 = (ann1 == true_labels).mean()
accuracy_2 = (ann2 == true_labels).mean()
accuracy_3 = (ann3 == true_labels).mean()
print(f'\nAccuracy vs gold standard:')
print(f'  Annotator 1: {accuracy_1:.3f}')
print(f'  Annotator 2: {accuracy_2:.3f}')
print(f'  Annotator 3: {accuracy_3:.3f}')